# 01 - Environment Setup
## FoodLens Databricks Pipeline
### Purpose: Create all schemas, volumes, metadata tables, and Gold layer DDL

In [0]:
    %run "./Util"

## Step 1: Create Schemas and Volumes

In [0]:
# Create Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")

# Create Schemas
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{META_SCHEMA}")

# Create Volumes
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{RAW_VOLUME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{STG_VOLUME}")

print("✅ Schemas and Volumes created!")

✅ Schemas and Volumes created!


## Step 2: Create Metadata Tables
Parent metadata drives which tables are active for ingestion.
Child metadata logs every pipeline run with status and row counts.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE} (
        table_name    STRING,
        file_name     STRING,
        active_flag   STRING,
        created_date  DATE,
        modified_date DATE
    )
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} (
        table_name       STRING,
        execution_time   TIMESTAMP,
        status           STRING,
        source_row_count LONG,
        target_row_count LONG,
        file_location    STRING,
        created_date     DATE
    )
""")
print("✅ Metadata tables created!")

✅ Metadata tables created!


## Step 3: Seed Parent Metadata
Insert active table entries so the pipeline knows what to process.

In [0]:
spark.sql(f"DELETE FROM {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE}")

spark.sql(f"""
    INSERT INTO {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE} VALUES
    ('Chicago', 'Food_Inspections_20260406.csv', 'Y', current_date(), current_date()),
    ('Dallas', 'Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260406.csv', 'Y', current_date(), current_date())
""")

spark.sql(f"SELECT * FROM {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE}").show()

+----------+--------------------+-----------+------------+-------------+
|table_name|           file_name|active_flag|created_date|modified_date|
+----------+--------------------+-----------+------------+-------------+
|   Chicago|Food_Inspections_...|          Y|  2026-04-21|   2026-04-21|
|    Dallas|Restaurant_and_Fo...|          Y|  2026-04-21|   2026-04-21|
+----------+--------------------+-----------+------------+-------------+



## Step 4: Create Gold Layer Tables (DDL)
Empty tables created here. Data is loaded in 04_Silver_to_Gold_Load.

In [0]:
# DIM_DATE
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{DIM_DATE} (
        date_key           INT NOT NULL,
        full_date          DATE,
        year               INT,
        month_name         VARCHAR(50),
        month_number       INT,
        day                VARCHAR(50),
        quarter            VARCHAR(50),
        week_number        INT,
        _date_to_warehouse TIMESTAMP NOT NULL,
        CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
    )
""")
print("✅ dim_date created!")

✅ dim_date created!


In [0]:
# DIM_RESTAURANT (SCD Type 2)
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT} (
        restaurant_key       BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
        restaurant_id        VARCHAR(255),
        dba_name             VARCHAR(500),
        aka_name             VARCHAR(500),
        facility_type        VARCHAR(255),
        address              VARCHAR(500),
        city                 VARCHAR(50),
        state                VARCHAR(50),
        zip_code             VARCHAR(10),
        change_hash          VARCHAR(50) NOT NULL,
        effective_start_date TIMESTAMP NOT NULL,
        effective_end_date   TIMESTAMP NOT NULL,
        is_current           BOOLEAN NOT NULL,
        _date_to_warehouse   TIMESTAMP NOT NULL,
        CONSTRAINT pk_dim_restaurant PRIMARY KEY (restaurant_key)
    )
""")
print("✅ dim_restaurant (SCD2) created!")

✅ dim_restaurant (SCD2) created!


In [0]:
# DIM_VIOLATION_DETAIL
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL} (
        violation_code_key    BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
        violation_id          VARCHAR(50) NOT NULL,
        violation_code        VARCHAR(50),
        violation_description VARCHAR(1000),
        violation_detail      VARCHAR(1000),
        _date_to_warehouse    TIMESTAMP NOT NULL,
        CONSTRAINT pk_dim_violation PRIMARY KEY (violation_code_key)
    )
""")
print("✅ dim_violation_detail created!")

✅ dim_violation_detail created!


In [0]:
# FACT_INSPECTIONS
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS} (
        inspection_key         BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
        inspection_id          VARCHAR(255) NOT NULL,
        inspection_date_key    INT,
        restaurant_key         BIGINT,
        license_num            INT,
        risk_category          VARCHAR(50),
        inspection_score       INT,
        inspection_result      VARCHAR(100),
        total_violation        INT,
        total_violation_points INT,
        inspection_type        VARCHAR(50),
        source_city            VARCHAR(50),
        pass_fail_flag         VARCHAR(10),
        _date_to_warehouse     TIMESTAMP NOT NULL,
        CONSTRAINT pk_fact_inspections PRIMARY KEY (inspection_key)
    )
""")
print("✅ fact_inspections created!")

✅ fact_inspections created!


In [0]:
# FACT_VIOLATIONS
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{FACT_VIOLATIONS} (
        violation_key      BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
        inspection_key     BIGINT,
        violation_code_key BIGINT,
        violation_points   INT,
        is_critical        VARCHAR(10),
        violation_comment  VARCHAR(10000),
        _date_to_warehouse TIMESTAMP NOT NULL,
        CONSTRAINT pk_fact_violations PRIMARY KEY (violation_key)
    )
""")
print("✅ fact_violations created!")

✅ fact_violations created!


In [0]:
# QUARANTINE TABLE
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{GOLD_QUARANTINE_INSPECTION} (
        inspection_id        VARCHAR(255),
        restaurant_id        VARCHAR(255),
        restaurant_name      VARCHAR(500),
        aka_name             VARCHAR(500),
        inspection_date      DATE,
        inspection_type      VARCHAR(100),
        inspection_result    VARCHAR(100),
        violation_score      INT,
        violation_code       VARCHAR(50),
        violation_points     INT,
        pass_fail_flag       VARCHAR(10),
        source_city          VARCHAR(50),
        address              VARCHAR(500),
        zip_code             VARCHAR(10),
        quarantine_category  VARCHAR(100),
        failed_column        VARCHAR(100),
        source_system        VARCHAR(100),
        pipeline_name        VARCHAR(100),
        _date_to_warehouse   TIMESTAMP,
        quarantine_timestamp TIMESTAMP
    )
""")
print("✅ Quarantine table created!")

✅ Quarantine table created!


## Step 5: Verify Setup

In [0]:
print("=" * 50)
print("SETUP COMPLETE — SUMMARY")
print("=" * 50)
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show()
spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").show()
spark.sql(f"SELECT * FROM {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE}").show()
print("✅ All done! Ready to run 02_Raw_to_Bronze_Load")

SETUP COMPLETE — SUMMARY
+------------------+
|      databaseName|
+------------------+
|       bronze_zone|
|           default|
|         gold_zone|
|information_schema|
| pipeline_metadata|
|       silver_zone|
+------------------+

+---------+--------------------+-----------+
| database|           tableName|isTemporary|
+---------+--------------------+-----------+
|gold_zone|            dim_date|      false|
|gold_zone|      dim_restaurant|      false|
|gold_zone|dim_violation_detail|      false|
|gold_zone|    fact_inspections|      false|
|gold_zone|     fact_violations|      false|
|gold_zone|quarantine_inspec...|      false|
+---------+--------------------+-----------+

+----------+--------------------+-----------+------------+-------------+
|table_name|           file_name|active_flag|created_date|modified_date|
+----------+--------------------+-----------+------------+-------------+
|   Chicago|Food_Inspections_...|          Y|  2026-04-21|   2026-04-21|
|    Dallas|Restauran